# FLUX.2 Klein Base 4B inference in Google Colab

This notebook loads the unmodified `black-forest-labs/FLUX.2-klein-base-4B` Diffusers snapshot from persistent Google Drive storage and generates images from text prompts. It does not use ARC conditioning, NVARC, LoRA, fine-tuning, or custom model components.

**Audience:** Diffusion-ARC development and lightweight prompt experiments in the Google Colab web interface.

**Prerequisites:** the complete model snapshot at `MyDrive/diffusion-arc/models/flux2-klein-base-4b` and a Colab GPU runtime. An A100 is the preferred fast path; smaller GPUs automatically use CPU offloading.

**Result:** the pipeline is loaded once, and the generation cell can be rerun with different prompts, seeds, resolutions, step counts, and guidance scales. Images and compact JSON metadata are saved to Drive.


## Outline

1. Check or install the pinned inference dependencies.
2. Mount Drive and validate the model snapshot and GPU.
3. Load the original pipeline once.
4. Edit and rerun one generation cell.


## 1. Python dependencies

The cell below installs only missing or mismatched inference libraries. It deliberately does not install or replace PyTorch, so Colab's CUDA runtime remains intact. The model weights are never downloaded by this notebook.


In [ ]:
from __future__ import annotations

import importlib.metadata
import subprocess
import sys

PINNED_PACKAGES = {
    "diffusers": "0.39.0",
    "transformers": "4.56.1",
    "accelerate": "1.12.0",
    "safetensors": "0.8.0",
}

packages_to_install = []
for distribution, expected_version in PINNED_PACKAGES.items():
    try:
        installed_version = importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        installed_version = None
    if installed_version != expected_version:
        packages_to_install.append(f"{distribution}=={expected_version}")

if packages_to_install:
    print("Installing:", " ".join(packages_to_install))
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            "--no-cache-dir",
            "--upgrade-strategy",
            "only-if-needed",
            *packages_to_install,
        ],
        check=True,
    )
else:
    print("All pinned inference packages are already installed.")

{name: importlib.metadata.version(name) for name in PINNED_PACKAGES}


## 2. Mount Drive and validate the runtime

Run this in the Colab web interface and approve the Drive authorization prompt. `OFFLOAD_MODE = "auto"` keeps the complete pipeline on an A100, while GPUs with less than 32 GiB automatically use CPU offloading. Set it to `"on"` or `"off"` only when you intentionally want to override that decision.


In [ ]:
import os
from pathlib import Path

from google.colab import drive

DRIVE_MOUNT = Path("/content/drive")
DRIVE_ROOT = DRIVE_MOUNT / "MyDrive" / "diffusion-arc"
MODEL_PATH = DRIVE_ROOT / "models" / "flux2-klein-base-4b"
OUTPUT_DIR = DRIVE_ROOT / "outputs" / "flux2-playground"
GPU_INDEX = 0
OFFLOAD_MODE = "auto"  # Choose from: "auto", "on", "off".
ENABLE_VAE_TILING = False

drive.mount(str(DRIVE_MOUNT), force_remount=False)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Refuse accidental model downloads: every component must come from Drive.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import torch

required_model_entries = (
    "model_index.json",
    "scheduler",
    "text_encoder",
    "tokenizer",
    "transformer",
    "vae",
)
assert MODEL_PATH.is_dir(), f"Model directory not found: {MODEL_PATH}"
missing_entries = [
    name for name in required_model_entries if not (MODEL_PATH / name).exists()
]
assert not missing_entries, f"Incomplete model snapshot; missing: {missing_entries}"
assert torch.cuda.is_available(), (
    "Select Runtime > Change runtime type > GPU before continuing."
)
assert GPU_INDEX < torch.cuda.device_count(), (GPU_INDEX, torch.cuda.device_count())
assert OFFLOAD_MODE in {"auto", "on", "off"}, OFFLOAD_MODE

torch.cuda.set_device(GPU_INDEX)
gpu_properties = torch.cuda.get_device_properties(GPU_INDEX)
gpu_memory_gib = gpu_properties.total_memory / 2**30
bf16_supported = bool(getattr(torch.cuda, "is_bf16_supported", lambda: False)())
TORCH_DTYPE = torch.bfloat16 if bf16_supported else torch.float16
USE_CPU_OFFLOAD = (
    OFFLOAD_MODE == "on"
    or (OFFLOAD_MODE == "auto" and gpu_memory_gib < 32.0)
)

runtime = {
    "model_path": str(MODEL_PATH),
    "gpu": torch.cuda.get_device_name(GPU_INDEX),
    "gpu_memory_GiB": round(gpu_memory_gib, 1),
    "dtype": str(TORCH_DTYPE),
    "cpu_offload": USE_CPU_OFFLOAD,
    "vae_tiling": ENABLE_VAE_TILING,
    "output_dir": str(OUTPUT_DIR),
}
runtime


## 3. Load the original pipeline

Run this cell once per Colab runtime. `local_files_only=True` and the offline environment variables ensure that every model component comes from Drive. Loading can take several minutes; generation cells reuse the loaded pipeline.


In [ ]:
from diffusers import Flux2KleinPipeline

pipe = Flux2KleinPipeline.from_pretrained(
    MODEL_PATH,
    torch_dtype=TORCH_DTYPE,
    local_files_only=True,
)

if ENABLE_VAE_TILING:
    pipe.vae.enable_tiling()

if USE_CPU_OFFLOAD:
    pipe.enable_model_cpu_offload(gpu_id=GPU_INDEX)
else:
    pipe.to(f"cuda:{GPU_INDEX}")

pipe.set_progress_bar_config(leave=False)
print(
    f"Loaded {pipe.__class__.__name__} from {MODEL_PATH} with "
    f"dtype={TORCH_DTYPE} and cpu_offload={USE_CPU_OFFLOAD}."
)


## 4. Generate and save images

Edit the parameters at the top, then rerun only this cell. The Base checkpoint defaults to 50 inference steps and guidance scale 4.0. Start at 512×512, then increase the resolution after the first successful run. Width and height must be divisible by 16.


In [ ]:
import json
import time
from datetime import datetime, timezone

from IPython.display import display

# Edit these parameters, then rerun only this cell.
PROMPT = (
    "A photorealistic soccer ball balanced on top of the crossbar of a football goal, "
    "empty stadium in the background, soft natural daylight, highly detailed"
)
SEED = 42
HEIGHT = 512
WIDTH = 512
NUM_INFERENCE_STEPS = 50
GUIDANCE_SCALE = 4.0
NUM_IMAGES_PER_PROMPT = 1
OUTPUT_STEM = "soccer_ball_on_goal_crossbar"

assert PROMPT.strip(), "PROMPT must not be empty."
assert HEIGHT > 0 and HEIGHT % 16 == 0, HEIGHT
assert WIDTH > 0 and WIDTH % 16 == 0, WIDTH
assert NUM_INFERENCE_STEPS > 0, NUM_INFERENCE_STEPS
assert GUIDANCE_SCALE >= 1.0, GUIDANCE_SCALE
assert 1 <= NUM_IMAGES_PER_PROMPT <= 4, NUM_IMAGES_PER_PROMPT
safe_output_stem = "".join(
    character if character.isalnum() or character in {"-", "_"} else "_"
    for character in OUTPUT_STEM.strip()
).strip("_")
assert safe_output_stem, "OUTPUT_STEM must contain letters or digits."

generators = [
    torch.Generator(device="cpu").manual_seed(SEED + index)
    for index in range(NUM_IMAGES_PER_PROMPT)
]
generator = generators[0] if len(generators) == 1 else generators

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats(GPU_INDEX)
torch.cuda.synchronize(GPU_INDEX)
started_at = time.perf_counter()
with torch.inference_mode():
    result = pipe(
        prompt=PROMPT,
        height=HEIGHT,
        width=WIDTH,
        num_inference_steps=NUM_INFERENCE_STEPS,
        guidance_scale=GUIDANCE_SCALE,
        num_images_per_prompt=NUM_IMAGES_PER_PROMPT,
        generator=generator,
    )
torch.cuda.synchronize(GPU_INDEX)
elapsed_seconds = time.perf_counter() - started_at
peak_gpu_memory_gib = torch.cuda.max_memory_reserved(GPU_INDEX) / 2**30

created_at = datetime.now(timezone.utc)
run_id = (
    f"{safe_output_stem}_{created_at.strftime('%Y%m%dT%H%M%S_%fZ')}_"
    f"seed{SEED}_{WIDTH}x{HEIGHT}"
)
saved_paths = []
for index, image in enumerate(result.images):
    output_path = OUTPUT_DIR / f"{run_id}_{index}.png"
    image.save(output_path)
    saved_paths.append(str(output_path))
    display(image)

generation_summary = {
    "created_at_utc": created_at.isoformat(),
    "model_path": str(MODEL_PATH),
    "prompt": PROMPT,
    "seeds": [SEED + index for index in range(NUM_IMAGES_PER_PROMPT)],
    "size": [WIDTH, HEIGHT],
    "steps": NUM_INFERENCE_STEPS,
    "guidance_scale": GUIDANCE_SCALE,
    "dtype": str(TORCH_DTYPE),
    "cpu_offload": USE_CPU_OFFLOAD,
    "elapsed_seconds": round(elapsed_seconds, 2),
    "peak_gpu_memory_GiB": round(peak_gpu_memory_gib, 2),
    "saved_paths": saved_paths,
}
metadata_path = OUTPUT_DIR / f"{run_id}.json"
generation_summary["metadata_path"] = str(metadata_path)
metadata_path.write_text(
    json.dumps(generation_summary, indent=2), encoding="utf-8"
)
generation_summary


## Variations, exercise, and common pitfalls

**Exercise:** keep the prompt fixed and compare seeds 42 and 43. Then keep seed 42 fixed and change only the guidance scale from 4.0 to 2.5. This separates composition changes caused by noise from changes caused by prompt guidance.

**Answer scaffold:** copy one of the alternatives below into the generation cell and rerun it.

Common pitfalls:

- A model-path assertion means Drive is not mounted or the download snapshot is incomplete.
- A CUDA assertion means the runtime is still using CPU; select an A100 or another GPU under **Runtime > Change runtime type**.
- On an A100, automatic mode keeps the model on the GPU. If memory is unexpectedly exhausted, set `OFFLOAD_MODE = "on"`, rerun the runtime cell, and reload the pipeline.
- Loading directly from Drive is slower than local SSD, but it happens only once per runtime. Reuse `pipe` for later generations.
- Four-step sampling is intended for distilled Klein checkpoints. This Base checkpoint intentionally defaults to 50 steps.
- To release memory completely, run `del pipe`, then `import gc; gc.collect(); torch.cuda.empty_cache()`.


In [ ]:
ALTERNATIVE_PROMPTS = {
    "product_photo": (
        "A bright red toy robot sitting on a wooden desk beside a small green plant, "
        "warm window light, realistic product photography"
    ),
    "illustration": (
        "A tiny observatory on a floating island above the clouds, a brass telescope "
        "pointing at a ringed planet, detailed storybook illustration"
    ),
}
ALTERNATIVE_PROMPTS
